# LANID annual irrigation maps 1997–2017 and derived layers over the Ogallala aquifer, on LANID's own 30 m grid

LANID (Landsat-based Irrigation Dataset, Xie, Gibbs and Lark 2021) maps irrigated cropland across the
conterminous US at 30 m for every year from 1997 to 2017. This notebook downloads the release from Zenodo,
cuts every layer to the aquifer on LANID's own grid, and writes one CF NetCDF file to Drive.

| | |
|---|---|
| Output | `MyDrive/MSUGWB/LANID/LANID_v3_30m_1997-2017_Ogallala.nc` |
| Source | Zenodo record 5548555, LANID-US version 3.0, CC-BY-4.0 (1.8 GB in all) |
| Grid | LANID's own Albers grid (EPSG:5070, 30 m; the same lattice as the CDL). **Nothing is resampled** |
| Cells kept | cells whose centre lies inside the aquifer polygon; the others hold the fill value |

**What is in the file**

* Root group: `irrigated(time, y, x)`, the 21 annual maps: 1 irrigated, 0 not irrigated (255 outside the
  aquifer), and `roi_mask`.
* Group `derived` (same 30 m grid): `irrFreq` (years irrigated, 1–21), `maxIrrExt` (irrigated at least once),
  `irrFreqChange` (years irrigated in 2008–2017 minus 1998–2007), `formerIrr`, `irrWholePeriod`,
  `intermittentIrr`, and `irrFreqPasture_West` (pasture/hay irrigation frequency, western US).
* Group `derived_irrAreaGain`: `irrAreaGain` (hectares gained from 1998–2007 to 2008–2017) on its own 6 km grid.

The meanings are those the Zenodo record gives. Each variable's attributes name the file it came from.

**How to run**

1. *Run all* with `TEST_MODE = True`: a 2048 × 2048 cell square in the middle of the aquifer, years 2016–2017
   and every derived layer, into `.../_test/` (about 1 GB is downloaded).
2. Set `TEST_MODE = False` and *Run all*. Running again does nothing if Drive has the file made from the same
   Zenodo files (`FORCE = True` makes it again).

**What the notebook checks**

* Every download must have the size and MD5 checksum Zenodo states; every GeoTIFF taken out of a zip must
  match the zip's own checksum.
* All annual maps must be on one grid; a derived layer on the same lattice is placed by a shift of whole
  cells, any other is kept on its own grid in a group of its own.
* An aquifer cell holding the fill value itself stops the run. Every strip written is read back from the
  closed file and compared before the file is copied to Drive, with a checksum comparison.

**Not included:** the 2018–2020 maps that USGS published separately (ScienceBase, doi:10.5066/P9YWR0OJ); they
need a browser download.

In [ ]:
# ═══════════════ CELL 1: install ═══════════════
# numpy, geopandas, shapely, pyproj and requests are part of Colab.
!pip install -q rasterio netCDF4 2>&1 | tail -2

In [ ]:
# ═══════════════ CELL 2: mount Drive, stage the shapefile, SETTINGS ═══════════════
import os, zipfile
if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

SHP_ZIP  = '/content/drive/MyDrive/high_plains_quifer.zip'
SHP_DIR  = '/content/ogallala_shp'
SHP_PATH = f'{SHP_DIR}/high_plains_quifer/hp_bound2010.shp'
if not os.path.exists(SHP_PATH):
    with zipfile.ZipFile(SHP_ZIP) as z:
        z.extractall(SHP_DIR)
print("Shapefile ready:", os.path.exists(SHP_PATH))

# ================= SETTINGS (edit these, then Run all) =================
TEST_MODE     = True       # True -> rehearsal into .../_test/ ; False -> the whole aquifer
TEST_YEARS    = [2016, 2017]   # annual maps of the rehearsal (one Zenodo zip is downloaded for them)
TEST_WINDOW_PX = 2048      # the rehearsal covers a square of this many 30 m pixels in the middle of the aquifer

ZENODO_RECORD = "5548555"  # LANID-US on Zenodo (version 3.0, CC-BY-4.0); file list and MD5 checksums come from its API
YEARS     = list(range(1997, 2018))   # annual maps lanid1997 ... lanid2017
DERIVED   = ["irrFreq", "maxIrrExt", "irrFreqChange", "formerIrr", "irrWholePeriod", "intermittentIrr",
             "irrFreqPasture_West", "irrAreaGain"]        # the derived layers of the record (irrAreaGain is on a 6 km grid)
PAD_CELLS = 2              # cells kept around the aquifer's bounding box
FORCE     = False          # True -> make the file again although Drive has it from the same Zenodo files

COMPRESS_LEVEL = 2         # zlib level of the .nc file (lossless)
CHUNK_YX       = 512
STRIP_ROWS     = 2048      # rows read, checked and written at a time
MIN_FREE_GB    = 10.0
DOWNLOAD_TRIES = 5
KEEP_DOWNLOADS = False
# =======================================================================

In [ ]:
# ═══════════════ CELL 3: imports, logging, paths, what each layer means ═══════════════
import re, gc, sys, json, glob, time, math, zlib, shutil, hashlib, logging, threading, warnings
from datetime import datetime, timezone

import numpy as np
import netCDF4 as nc4
import geopandas as gpd
import pyproj
import rasterio
import rasterio.features
import rasterio.windows
from affine import Affine
import requests

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-7s | %(message)s",
                    datefmt="%H:%M:%S", force=True)
log = logging.getLogger("lanid")
for _n in ("urllib3.connectionpool", "rasterio", "pyogrio"):
    logging.getLogger(_n).setLevel(logging.ERROR)

ROI_LABEL   = "Ogallala"
PRODUCT_TAG = "LANID_v3_30m"
DRIVE_OUT   = "/content/drive/MyDrive/MSUGWB/LANID" + ("/_test" if TEST_MODE else "")
OUT_NAME    = f"{PRODUCT_TAG}_1997-2017_{ROI_LABEL}.nc"
SCRATCH     = "/content/lanid_scratch" + ("_test" if TEST_MODE else "")
RAW_DIR     = f"{SCRATCH}/raw"
NC_DIR      = f"{SCRATCH}/nc"
DISK_PATH   = "/content"
for _d in (DRIVE_OUT, RAW_DIR, NC_DIR):
    os.makedirs(_d, exist_ok=True)

ZENODO_API  = f"https://zenodo.org/api/records/{ZENODO_RECORD}"
HTTP = requests.Session()
HTTP.headers["User-Agent"] = "MSU groundwater project LANID notebook (python-requests)"

# Meanings as the Zenodo record states them (Xie, Gibbs and Lark 2021, ESSD 13, 5689).
MEANING = dict(
    irrigated="annual irrigation map: 1 irrigated, 0 not irrigated (LANID lanidYYYY)",
    irrFreq="number of years irrigated during 1997-2017 (1-21)",
    maxIrrExt="croplands irrigated at least once during 1997-2017 (binary)",
    irrFreqChange="number of years irrigated in 2008-2017 minus that in 1998-2007",
    formerIrr="not irrigated anytime in 2015-2017, but irrigated at least 3 times before (binary)",
    irrWholePeriod="irrigated at least once in both 1997-1999 and 2015-2017 (binary)",
    intermittentIrr="irrigated at least once in both 1997-1999 and 2015-2017, and irrigation frequency <= 18 (binary)",
    irrFreqPasture_West="irrigation frequency of pasture/hay, western CONUS",
    irrAreaGain="LANID-derived irrigation gain (hectares) from 1998-2007 to 2008-2017, on a 6 km x 6 km grid")
UNITS = dict(irrFreq="years", irrFreqChange="years", irrFreqPasture_West="years", irrAreaGain="ha")


class PipelineStop(RuntimeError):
    """Clean stop: nothing on Drive was changed; re-running resumes."""


class DataIntegrityError(RuntimeError):
    """Something that must never reach the output was detected."""


def utcnow():
    return datetime.now(timezone.utc).replace(tzinfo=None)


def iso(dt):
    return dt.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"

In [ ]:
# ═══════════════ CELL 4: disk floor, verified Drive copy, logs ═══════════════
_LOG_LOCK   = threading.Lock()
_DRIVE_LOCK = threading.Lock()
DRIVE_SETTLE_S = 10                      # let the Drive FUSE mount settle before re-reading
TIMES = {}                               # seconds spent per stage (reported at the end of the run)
TOTALS = {}                              # counters (bytes downloaded, files, records) for the summary


class timed:
    """with timed("download"): ...  adds the elapsed seconds to TIMES["download"]."""

    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t0 = time.time()
        return self

    def __exit__(self, *exc):
        TIMES[self.name] = TIMES.get(self.name, 0.0) + time.time() - self.t0
        return False


def log_events(name, entries):
    """Append JSON lines to a log on Drive in one write + fsync (survives a dead runtime)."""
    if not entries:
        return
    stamp = iso(utcnow())
    text = "".join(json.dumps(dict(e, logged=stamp), default=str) + "\n" for e in entries)
    with _LOG_LOCK:
        with open(os.path.join(DRIVE_OUT, name), "a") as f:
            f.write(text)
            f.flush()
            os.fsync(f.fileno())


def human(n):
    n = float(n)
    for u in ("B", "KB", "MB", "GB", "TB"):
        if abs(n) < 1024:
            return f"{n:.1f} {u}"
        n /= 1024
    return f"{n:.1f} PB"


# ---------- 20 GB scratch floor ----------
def free_gb():
    return shutil.disk_usage(DISK_PATH).free / 1024 ** 3


def ensure_disk_free(context="", extra_gb=0.0):
    """Block until MIN_FREE_GB + extra_gb is free. Scratch only ever holds the file being built,
    so a shortfall normally means Drive's local upload cache is still draining: wait for it
    rather than fail. Raises PipelineStop if it never clears."""
    need = MIN_FREE_GB + max(0.0, extra_gb)
    t0, warned = time.time(), False
    while True:
        free = free_gb()
        if free >= need:
            if warned:
                log.info("Disk OK again: %.1f GB free (need %.1f)", free, need)
            return free
        if not warned:
            warned = True
            log.warning("LOW DISK %.1f < %.1f GB (%s) - waiting for Drive to upload", free, need, context)
        if time.time() - t0 > DISK_MAX_WAIT_S:
            raise PipelineStop(f"Free space stayed at {free:.1f} GB (< {need:.1f} GB) for "
                               f"{DISK_MAX_WAIT_S // 60} min ({context}). Nothing is lost - re-run to resume.")
        time.sleep(60)


# ---------- verified copies to / from Drive ----------
def md5sum(path, block=10 * 1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(block)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def robust_drive_copy(src, dst, max_retries=3):
    """Copy src to Drive under a temporary name, re-read it, compare size + MD5, then
    rename over dst. A runtime that dies mid-copy leaves the previous good dst intact."""
    if os.path.abspath(src) == os.path.abspath(dst):
        raise ValueError(f"robust_drive_copy: source and destination are the same file ({src})")
    with _DRIVE_LOCK, timed("Drive copy"):
        size = os.path.getsize(src)
        ensure_disk_free("Drive copy", extra_gb=size / 1024 ** 3)   # FUSE caches the upload locally
        want = md5sum(src)
        tmp = dst + ".tmp"
        os.makedirs(os.path.dirname(dst) or ".", exist_ok=True)
        last = None
        for attempt in range(1, max_retries + 1):
            try:
                with open(src, "rb") as fs, open(tmp, "wb") as fd:
                    shutil.copyfileobj(fs, fd, length=10 * 1024 * 1024)
                    fd.flush()
                    os.fsync(fd.fileno())
                ok = False
                for _ in range(4):           # a just-closed FUSE file can return stale bytes
                    time.sleep(DRIVE_SETTLE_S)
                    try:
                        if os.path.getsize(tmp) == size and md5sum(tmp) == want:
                            ok = True
                            break
                    except OSError as e:
                        last = e
                if not ok:
                    raise IOError("size or MD5 mismatch after settle")
                os.replace(tmp, dst)
                log.info("Drive copy verified (%s): %s", human(size), os.path.basename(dst))
                return True
            except Exception as e:
                last = e
                log.warning("Drive copy attempt %d/%d failed: %s", attempt, max_retries, e)
                try:
                    if os.path.exists(tmp):
                        os.remove(tmp)
                except OSError:
                    pass
                if attempt < max_retries:
                    time.sleep(10 * attempt)
        raise RuntimeError(f"Drive copy of {os.path.basename(src)} failed after {max_retries} attempts: {last}")


def restore_from_drive(src, dst):
    """Bring a Drive file back to scratch (size-checked, atomic)."""
    size = os.path.getsize(src)
    ensure_disk_free("restore from Drive", extra_gb=size / 1024 ** 3)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    tmp = dst + ".part"
    shutil.copyfile(src, tmp)
    if os.path.getsize(tmp) != size:
        os.remove(tmp)
        raise IOError(f"restore of {os.path.basename(src)} is incomplete")
    os.replace(tmp, dst)
    log.info("Restored %s from Drive (%s)", os.path.basename(src), human(size))

In [ ]:
# ═══════════════ CELL 5: Zenodo's files: list, download (MD5-checked), take out the GeoTIFFs ═══════════════
def zenodo_files():
    """{file name: dict(url, size, md5)} of the Zenodo record, from its API."""
    for attempt in range(DOWNLOAD_TRIES):
        try:
            r = HTTP.get(ZENODO_API, timeout=60)
            r.raise_for_status()
            rec = r.json()
            break
        except (IOError, requests.RequestException) as e:
            log.warning("Zenodo's record could not be read (%s) - trying again", str(e)[:120])
            time.sleep(30 * (attempt + 1))
    else:
        raise PipelineStop("Zenodo could not be reached. Nothing was changed - run again later.")
    out = {}
    for f in rec.get("files", []):
        algo, _, value = str(f.get("checksum", "")).partition(":")
        out[f["key"]] = dict(url=f["links"]["self"], size=int(f["size"]), md5=value if algo == "md5" else None)
    log.info("Zenodo record %s, version %s: %d files", ZENODO_RECORD, rec["metadata"].get("version"), len(out))
    return out, rec["metadata"].get("version")


def md5sum(path, block=16 * 1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(block)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


VERIFIED = set()


def download(name, info):
    """A Zenodo file on scratch, complete: its size and MD5 must equal Zenodo's. A break is continued
    with a byte range; a file that fails its MD5 is fetched again from the start."""
    path = os.path.join(RAW_DIR, name)
    if path in VERIFIED:
        return path
    if os.path.exists(path) and os.path.getsize(path) == info["size"] and (not info["md5"] or md5sum(path) == info["md5"]):
        VERIFIED.add(path)
        return path
    ensure_disk_free(f"download {name}", extra_gb=info["size"] / 1024 ** 3)
    part, t0 = path + ".part", time.time()
    for attempt in range(1, DOWNLOAD_TRIES + 1):
        have = os.path.getsize(part) if os.path.exists(part) else 0
        try:
            hdr = {"Range": f"bytes={have}-"} if have else {}
            with HTTP.get(info["url"], headers=hdr, stream=True, timeout=120) as r:
                if have and r.status_code != 206:
                    have = 0
                elif r.status_code not in (200, 206):
                    raise IOError(f"HTTP {r.status_code}")
                with open(part, "ab" if have else "wb") as f:
                    for chunk in r.iter_content(8 * 1024 * 1024):
                        f.write(chunk)
            if os.path.getsize(part) != info["size"]:
                raise IOError(f"{os.path.getsize(part)} of {info['size']} bytes")
            if info["md5"] and md5sum(part) != info["md5"]:
                os.remove(part)
                raise IOError("MD5 differs from Zenodo's")
            os.replace(part, path)
            VERIFIED.add(path)
            TOTALS["bytes"] = TOTALS.get("bytes", 0) + info["size"]
            log.info("Downloaded %s: %s in %.0f s (size and MD5 as Zenodo states)", name, human(info["size"]), time.time() - t0)
            return path
        except (IOError, requests.RequestException) as e:
            log.warning("Download of %s broke off (%s), attempt %d of %d", name, str(e)[:120], attempt, DOWNLOAD_TRIES)
            time.sleep(min(30 * attempt, 300))
    raise PipelineStop(f"{name} could not be downloaded completely. Nothing on Drive was changed - run again.")


def where_is(files):
    """{layer: (Zenodo file, member or None)} for every annual map and derived layer, from the file names:
    lanidA-B.zip hold lanidA.tif ... lanidB.tif, irrFreqAndChange.zip the derived layers, and a .tif of the
    record is a layer by itself. The members are confirmed when the zips are opened."""
    out = {}
    for name in files:
        m = re.fullmatch(r"lanid(\d{4})-(\d{4})\.zip", name)
        if m:
            for y in range(int(m.group(1)), int(m.group(2)) + 1):
                out[f"lanid{y}"] = (name, f"lanid{y}.tif")
        elif name == "irrFreqAndChange.zip":
            for d in DERIVED:
                out.setdefault(d, (name, f"{d}.tif"))
        elif name.endswith(".tif"):
            out[name[:-4]] = (name, None)
    return out


def take_out(zip_path, member):
    """One GeoTIFF taken out of a zip to scratch; zipfile checks it against the CRC stored in the zip."""
    out = os.path.join(RAW_DIR, member)
    with zipfile.ZipFile(zip_path) as z:
        try:
            info = z.getinfo(member)
        except KeyError:
            raise PipelineStop(f"{os.path.basename(zip_path)} holds no {member}; it holds {z.namelist()}")
        if os.path.exists(out) and os.path.getsize(out) == info.file_size:
            return out
        ensure_disk_free("unzip", extra_gb=info.file_size / 1024 ** 3)
        try:
            with z.open(info) as src, open(out + ".part", "wb") as dst:
                shutil.copyfileobj(src, dst, 16 * 1024 * 1024)
        except zipfile.BadZipFile as e:
            os.remove(out + ".part")
            raise PipelineStop(f"{member} in {os.path.basename(zip_path)} fails the zip's own check ({e}). Delete "
                               f"{zip_path} and run again.")
        os.replace(out + ".part", out)
    return out


def fetch_sources(years, derived):
    """Every needed layer as a GeoTIFF on scratch: {layer: path}, plus the record's version and checksums."""
    files, version = zenodo_files()
    loc = where_is(files)
    want = [f"lanid{y}" for y in years] + list(derived)
    missing = [k for k in want if k not in loc]
    if missing:
        raise PipelineStop(f"The Zenodo record holds no file for {missing}.")
    paths, used = {}, {}
    for layer in want:
        name, member = loc[layer]
        z = download(name, files[name])
        used[name] = files[name]["md5"]
        paths[layer] = take_out(z, member) if member else z
    if not KEEP_DOWNLOADS:
        for name in used:
            p = os.path.join(RAW_DIR, name)
            if p.endswith(".zip") and os.path.exists(p):
                os.remove(p)
    return paths, version, used

In [ ]:
# ═══════════════ CELL 6: the aquifer window on LANID's own grid; write the file strip by strip; check it ═══════════════
def aquifer_polygon(crs):
    gdf = gpd.read_file(SHP_PATH)
    if gdf.crs is None:
        raise PipelineStop(f"{SHP_PATH} has no CRS")
    g = gdf.geometry
    u = g.union_all() if hasattr(g, "union_all") else g.unary_union
    p = gpd.GeoSeries([u], crs=gdf.crs).to_crs(crs).iloc[0]
    return p if p.is_valid else p.buffer(0)


def grid_of(path):
    with rasterio.open(path) as ds:
        if ds.count != 1 or ds.transform.b or ds.transform.d:
            raise PipelineStop(f"{os.path.basename(path)}: one band on an unrotated grid was expected")
        return dict(crs=ds.crs, transform=ds.transform, w=ds.width, h=ds.height, dtype=ds.dtypes[0], nodata=ds.nodata)


def window_on(G, polygon, pad, square=False):
    """The block of G's cells that bounds the polygon (plus `pad` cells): (transform, h, w). With
    square=True (the rehearsal) a TEST_WINDOW_PX square in the middle of that block."""
    T = G["transform"]
    x0, y0, x1, y1 = polygon.bounds
    c0 = math.floor((x0 - T.c) / T.a) - pad
    c1 = math.ceil((x1 - T.c) / T.a) + pad
    r0 = math.floor((y1 - T.f) / T.e) - pad
    r1 = math.ceil((y0 - T.f) / T.e) + pad
    if square:
        cm, rm, n = (c0 + c1) // 2, (r0 + r1) // 2, TEST_WINDOW_PX
        c0, c1, r0, r1 = cm - n // 2, cm + n - n // 2, rm - n // 2, rm + n - n // 2
    return T * Affine.translation(c0, r0), r1 - r0, c1 - c0


def on_same_lattice(G, H):
    """True when H's pixels coincide with G's (same CRS and size, origins whole pixels apart): a layer
    on such a grid is read into G's window by a shift of whole pixels, never resampled."""
    a, b = G["transform"], H["transform"]
    dc, dr = (b.c - a.c) / a.a, (b.f - a.f) / a.e
    return (G["crs"] == H["crs"] and a.a == b.a and a.e == b.e
            and abs(dc - round(dc)) < 1e-6 and abs(dr - round(dr)) < 1e-6)


def fill_for(dtype, nodata):
    """The fill of a variable: the layer's own no-data value, else a value outside what the type
    holds in LANID (checked strip by strip while writing)."""
    if nodata is not None:
        return np.asarray(nodata).astype(dtype)[()]
    kind = np.dtype(dtype)
    if kind.kind == "f":
        return kind.type(np.nan)
    return np.iinfo(kind).max if kind.kind == "u" else np.iinfo(kind).min


def read_strip(path, WT, r0, r1, w, fill):
    """Rows r0..r1 of the window WT (w columns wide) from a layer on the same lattice, by whole-pixel
    shift; parts of the window the layer does not cover hold `fill`."""
    with rasterio.open(path) as ds:
        col0 = round((WT.c - ds.transform.c) / ds.transform.a)
        row0 = round((WT.f - ds.transform.f) / ds.transform.e) + r0
        win = rasterio.windows.Window(col0, row0, w, r1 - r0)
        return ds.read(1, window=win, boundless=True, fill_value=fill)


def strip_mask(polygon, WT, r0, r1, w):
    return rasterio.features.geometry_mask([polygon], out_shape=(r1 - r0, w), transform=WT * Affine.translation(0, r0),
                                           invert=True, all_touched=False)          # cell centre inside the polygon


def put_coords(ds, crs, T, h, w, prefix=""):
    yv = ds.createVariable("y", "f8", ("y",))
    yv.standard_name, yv.units, yv.axis, yv.long_name = "projection_y_coordinate", "m", "Y", "y of cell centre"
    yv[:] = T.f + (np.arange(h) + 0.5) * T.e
    xv = ds.createVariable("x", "f8", ("x",))
    xv.standard_name, xv.units, xv.axis, xv.long_name = "projection_x_coordinate", "m", "X", "x of cell centre"
    xv[:] = T.c + (np.arange(w) + 0.5) * T.a
    gm = ds.createVariable("crs", "i4")
    c = pyproj.CRS.from_wkt(crs.to_wkt())
    for k, v in c.to_cf().items():
        gm.setncattr(k, v if isinstance(v, (str, int, float)) else np.asarray(v) if isinstance(v, (list, tuple)) else str(v))
    gm.spatial_ref = c.to_wkt()
    gm.GeoTransform = f"{T.c} {T.a} 0.0 {T.f} 0.0 {T.e}"


def write_file(path, paths, years, derived, version, used):
    """The whole file: annual maps in the root group, the 30 m derived layers in the group 'derived', and
    layers on another grid (the 6 km irrAreaGain) in a group of their own. Returns checksums of every
    strip that was written, for the read-back check."""
    G = grid_of(paths[f"lanid{years[0]}"])
    polygon = aquifer_polygon(G["crs"])
    WT, h, w = window_on(G, polygon, PAD_CELLS, square=TEST_MODE)
    zkw = dict(zlib=COMPRESS_LEVEL > 0, complevel=max(COMPRESS_LEVEL, 1), shuffle=True)
    ch = (min(CHUNK_YX, h), min(CHUNK_YX, w))
    crcs, other = {}, []
    grids = {k: grid_of(p) for k, p in paths.items()}
    for k, H in grids.items():
        if k.startswith("lanid") and not on_same_lattice(G, H):
            raise PipelineStop(f"{k} is not on the grid of lanid{years[0]}; nothing was written.")
    tmp = path + ".new"
    with nc4.Dataset(tmp, "w", format="NETCDF4") as ds:
        ds.createDimension("time", len(years))
        ds.createDimension("y", h)
        ds.createDimension("x", w)
        tv = ds.createVariable("time", "i4", ("time",))
        tv.long_name, tv.units = "year of the annual map", "year"
        tv[:] = np.asarray(years, dtype="i4")
        put_coords(ds, G["crs"], WT, h, w)
        mv = ds.createVariable("roi_mask", "i1", ("y", "x"), chunksizes=ch, **zkw)
        mv.long_name = f"1 where the cell centre lies inside the {ROI_LABEL} aquifer polygon"
        mv.flag_values, mv.flag_meanings, mv.grid_mapping = np.array([0, 1], dtype="i1"), "outside_roi inside_roi", "crs"
        av = ds.createVariable("irrigated", "u1", ("time", "y", "x"), fill_value=np.uint8(255), chunksizes=(1,) + ch, **zkw)
        av.long_name, av.units, av.grid_mapping = MEANING["irrigated"], "1", "crs"
        av.flag_values, av.flag_meanings = np.array([0, 1], dtype="u1"), "not_irrigated irrigated"
        av.source_files = " ".join(f"lanid{y}.tif" for y in years)
        av.comment = "Exactly LANID's values on its own 30 m grid. 255: outside the aquifer polygon."
        dg = ds.createGroup("derived")
        dg.description = "Derived layers of LANID 1997-2017 on the same 30 m grid as the annual maps (see the root group's x, y, crs)"
        dvars = {}
        for name in derived:
            H = grids[name]
            if not on_same_lattice(G, H):
                other.append(name)
                continue
            fill = fill_for(H["dtype"], H["nodata"])
            v = dg.createVariable(name, H["dtype"], ("y", "x"), fill_value=fill, chunksizes=ch, **zkw)
            v.long_name, v.grid_mapping = MEANING.get(name, name), "/crs"
            v.units = UNITS.get(name, "1")
            v.source_file = os.path.basename(paths[name])
            v.comment = ("Exactly LANID's values. _FillValue: outside the aquifer polygon" +
                         (", or LANID's own no-data value." if H["nodata"] is not None else
                          ", or outside the area this layer covers. LANID gives this layer no no-data value; "
                          "0 is one of its values."))
            dvars[name] = (v, fill)
        # ---------- strips of rows: annual maps and 30 m derived layers ----------
        for r0 in range(0, h, STRIP_ROWS):
            r1 = min(h, r0 + STRIP_ROWS)
            m = strip_mask(polygon, WT, r0, r1, w)
            mv[r0:r1, :] = m.astype("i1")
            crcs[("roi_mask", None, r0)] = zlib.crc32(np.ascontiguousarray(m.astype("i1")))
            for t, y in enumerate(years):
                a = read_strip(paths[f"lanid{y}"], WT, r0, r1, w, 255)
                if np.count_nonzero(a[m] == 255):
                    raise DataIntegrityError(f"lanid{y}: an aquifer pixel holds 255 (outside the map, or a value LANID "
                                             "does not use); nothing was copied to Drive")
                a = np.where(m, a, 255).astype("u1")
                av[t, r0:r1, :] = a
                crcs[("irrigated", t, r0)] = zlib.crc32(np.ascontiguousarray(a))
            for name, (v, fill) in dvars.items():
                H = grids[name]
                a = read_strip(paths[name], WT, r0, r1, w, fill)
                if H["nodata"] is None and np.dtype(H["dtype"]).kind != "f":
                    inside = read_strip(paths[name], WT, r0, r1, w, 0)
                    if np.count_nonzero((inside == fill) & m):
                        raise DataIntegrityError(f"{name} holds the value {fill} itself, which this file uses as fill")
                a = np.where(m, a, fill).astype(H["dtype"])
                v[r0:r1, :] = a
                crcs[(f"derived/{name}", None, r0)] = zlib.crc32(np.ascontiguousarray(a))
            log.info("rows %d-%d of %d written", r0, r1, h)
        # ---------- layers on another grid: a group each, on their own grid, cut to the aquifer ----------
        for name in other:
            H = grids[name]
            poly_h = aquifer_polygon(H["crs"])
            T2, h2, w2 = window_on(H, poly_h, 1)
            g = ds.createGroup(f"derived_{name}")
            g.description = (f"{name}: on its own grid (cell {H['transform'].a:g} m), not on the 30 m grid of the root group")
            g.createDimension("y", h2)
            g.createDimension("x", w2)
            put_coords(g, H["crs"], T2, h2, w2)
            fill = fill_for(H["dtype"], H["nodata"])
            m2 = strip_mask(poly_h, T2, 0, h2, w2)
            mk = g.createVariable("roi_mask", "i1", ("y", "x"))
            mk.long_name = f"1 where the cell centre lies inside the {ROI_LABEL} aquifer polygon"
            mk[:] = m2.astype("i1")
            v = g.createVariable(name, H["dtype"], ("y", "x"), fill_value=fill, **zkw)
            v.long_name, v.units, v.grid_mapping = MEANING.get(name, name), UNITS.get(name, "1"), "crs"
            v.source_file = os.path.basename(paths[name])
            v.comment = "Exactly LANID's values on its own grid. _FillValue: LANID's own no-data value or outside the polygon."
            a = np.where(m2, read_strip(paths[name], T2, 0, h2, w2, fill), fill).astype(H["dtype"])
            v[:] = a
            crcs[(f"derived_{name}/{name}", None, 0)] = zlib.crc32(np.ascontiguousarray(a))
        ds.Conventions = "CF-1.9"
        ds.title = f"LANID annual irrigation maps 1997-2017 and derived layers, {ROI_LABEL} aquifer, 30 m"
        ds.summary = ("Landsat-based Irrigation Dataset (LANID-US): annual 30 m irrigation maps for 1997-2017 and the "
                      "derived layers, cut to the aquifer on LANID's own Albers grid. Nothing is resampled, reprojected "
                      "or filled; cells outside the aquifer polygon hold the fill value.")
        ds.source = f"Zenodo record {ZENODO_RECORD} (LANID-US version {version}), https://doi.org/10.5281/zenodo.{ZENODO_RECORD}"
        ds.references = ("Xie, Gibbs and Lark 2021, ESSD 13, 5689, https://doi.org/10.5194/essd-13-5689-2021; "
                         "Xie and Lark 2021, RSE 260, 112445, https://doi.org/10.1016/j.rse.2021.112445")
        ds.license = "CC-BY-4.0"
        ds.history = f"{iso(utcnow())} created by LANID_Irrigation_30m_Ogallala.ipynb"
        ds.source_md5 = json.dumps(used, sort_keys=True)
        ds.years = " ".join(map(str, years))
        ds.roi = ROI_LABEL
        ds.processing_status = "complete"
    os.replace(tmp, path)
    return crcs, (WT, h, w)


def check_file(path, crcs):
    """Every strip written is read back from the closed file and compared with its checksum."""
    with nc4.Dataset(path) as ds:
        ds.set_auto_maskandscale(False)
        for (name, t, r0), want in crcs.items():
            grp, _, var = name.rpartition("/")
            v = (ds[grp] if grp else ds).variables[var]
            if t is not None:
                got = v[t, r0:r0 + STRIP_ROWS, :]
            elif grp.startswith("derived_"):
                got = v[:]
            else:
                got = v[r0:r0 + STRIP_ROWS, :]
            if zlib.crc32(np.ascontiguousarray(np.asarray(got))) != want:
                raise DataIntegrityError(f"{name} rows {r0}+ read back differently from what was written")
    log.info("%d strips read back from the file, all identical to what was written", len(crcs))

In [ ]:
# ═══════════════ CELL 7: run ═══════════════
def already_done(path, used_md5):
    """True when Drive holds a complete file made from Zenodo files with these same checksums."""
    if FORCE or not os.path.exists(path):
        return False
    for attempt in range(3):
        try:
            with nc4.Dataset(path) as ds:
                return (str(getattr(ds, "processing_status", "")) == "complete"
                        and json.loads(str(ds.source_md5)) == used_md5)
        except Exception as e:
            why = e
            time.sleep(20 * (attempt + 1))
    raise PipelineStop(f"{os.path.basename(path)} is on Drive but cannot be read ({why}). Nothing was changed. If it "
                       "really is damaged, move it out of the folder yourself and run again.")


def run():
    t0 = time.time()
    TOTALS.clear()
    years = sorted(TEST_YEARS) if TEST_MODE else YEARS
    dst = os.path.join(DRIVE_OUT, OUT_NAME)
    try:
        ensure_disk_free("start")
        files, version = zenodo_files()
        loc = where_is(files)
        used = {loc[k][0]: files[loc[k][0]]["md5"] for k in [f"lanid{y}" for y in years] + DERIVED if k in loc}
        if already_done(dst, used):
            log.info("%s on Drive was made from these same Zenodo files - nothing to do (FORCE = True makes it again)", OUT_NAME)
            return dst
        paths, version, used = fetch_sources(years, DERIVED)
        local = os.path.join(NC_DIR, OUT_NAME)
        crcs, (WT, h, w) = write_file(local, paths, years, DERIVED, version, used)
        check_file(local, crcs)
        robust_drive_copy(local, dst)
        log.info("DONE: %s, %d x %d cells, %d annual maps and %d derived layers, %s on Drive, %.1f min", OUT_NAME, h, w,
                 len(years), len(DERIVED), human(os.path.getsize(local)), (time.time() - t0) / 60)
        os.remove(local)
        if not KEEP_DOWNLOADS:
            shutil.rmtree(RAW_DIR, ignore_errors=True)
        return dst
    except PipelineStop as e:
        print("\n" + "=" * 78 + f"\n  STOPPED (nothing on Drive was changed)\n  {e}\n" + "=" * 78 + "\n")
        return None
    finally:
        log.info("this run: %s downloaded from Zenodo, %.1f min", human(TOTALS.get("bytes", 0)), (time.time() - t0) / 60)


RESULT = run()

In [ ]:
# ═══════════════ CELL 8: what is in the file ═══════════════
def describe(path):
    if not path or not os.path.exists(path):
        print("No output file.")
        return
    with nc4.Dataset(path) as ds:
        ds.set_auto_maskandscale(False)
        inside = np.asarray(ds["roi_mask"][:]).astype(bool)
        n_in = int(inside.sum())
        print(f"{path}\n{human(os.path.getsize(path))} | {len(ds.dimensions['y'])} x {len(ds.dimensions['x'])} cells of 30 m | "
              f"{n_in:,} inside the aquifer\n\nirrigated share of the aquifer, by year:")
        for t, y in enumerate(ds["time"][:]):
            a = np.asarray(ds["irrigated"][t])[inside]
            print(f"  {int(y)}: {100 * np.count_nonzero(a == 1) / max(1, n_in):5.2f} %")
        for gname, g in ds.groups.items():
            print(f"\ngroup {gname}:")
            for name, v in g.variables.items():
                if v.dimensions != ("y", "x") or name == "roi_mask":
                    continue
                a = np.asarray(v[:])
                m = np.asarray(g["roi_mask"][:]).astype(bool) if "roi_mask" in g.variables else inside
                a = a[m]
                fill = getattr(v, "_FillValue", None)
                ok = (a != fill) & np.isfinite(a.astype("f8")) if fill is not None else np.isfinite(a.astype("f8"))
                vals = a[ok].astype("f8")
                rng = f"{vals.min():g} .. {vals.max():g}" if vals.size else "-"
                print(f"  {name:20s} {str(v.dtype):8s} values {rng:>18s} | {vals.size:,} cells with a value | {v.long_name}")


describe(RESULT or os.path.join(DRIVE_OUT, OUT_NAME))

# Reading the file:
#   import xarray as xr
#   path = "/content/drive/MyDrive/MSUGWB/LANID/LANID_v3_30m_1997-2017_Ogallala.nc"
#   annual  = xr.open_dataset(path, chunks={"time": 1, "y": 2048, "x": 2048})     # irrigated(time, y, x)
#   derived = xr.open_dataset(path, group="derived", chunks={"y": 2048, "x": 2048})
#   freq    = derived.irrFreq                       # same y and x as the annual maps
#   gain    = xr.open_dataset(path, group="derived_irrAreaGain")                  # its own 6 km grid
# The grid is LANID's own (EPSG:5070, 30 m), the same lattice as the CDL's.